![Platform+](images/banner.png)

# LAB 6: MARKETPLACE
## Query data you don't own — zero copy, no pipeline

> **Persona track: Data Analyst** — same lab, your angle.

:point_right: **In this lesson, you'll access a Private Listing shared with your account, explore it without loading a single row, and query it to help Roastery & Co. plan their café reading corner.**

The scenario: Roastery & Co. is opening a reading corner in their cafés and wants to stock books that resonate with their customers. A fictional publisher — **Alpine Peaks Publishing** — has shared their forthcoming catalog privately with your Snowflake Education account. The data is already in your account as a zero-copy share: no pipeline, no loading, just queries.

We'll cover:
1. Two ways to get data: loading vs. querying in place
2. Public Marketplace vs. Private Listing — and when each fits
3. Searching the public Marketplace — by hand and with CoCo's skills
4. Getting access to a privately shared database
5. Building a reusable view over the shared catalog for the café

:books: **By the end you'll be able to:**
- Explain what "zero copy" means and when it's the right choice
- Distinguish Public Marketplace listings from Private Listings
- Search the public Marketplace by hand and with CoCo's `/marketplace-search` and `/get-marketplace-listing-details` skills
- Grant your role access to a privately shared database
- Build a reusable view over a shared table and query it without ever loading a row

**Time:** ~15 min · **Lab interface:** Workspace Notebook with Cortex Code (CoCo) as your tutor.
*Source data lives in the shared, read-only `PLATPLUS_DB` and in the `ALPINE_PEAKS_PUBLISHING` listing shared privately with this account.*

---
👉 **Connecting to your compute service:**
1. In the top bar, if no service is connected, click to connect and select your service.
2. Set your **role and warehouse** using the picker at the top-right — or just run the setup cell below, which sets them for you in SQL. (SQL cells run on the warehouse.)
3. Once the status shows **Connected** (green), run a cell with **Shift+Enter** (run + advance) or **Cmd/Ctrl+Enter** (run this cell only); the **▶** button works too.

> 📌 The service is a container that runs the notebook kernel and your **Python** cells. **SQL cells (and Snowpark pushdown) run on the query warehouse**, not the container. It stays running until it hits its idle timeout or you suspend it manually.
> 💡 **Best practice: Start a fresh chat for this lab.** Before you begin, open a fresh CoCo chat — click **New chat** (the **+** at the top of the panel). Each lab creates its own objects, so a fresh chat ensures CoCo isn't referencing tables or schemas from a prior lab that no longer exist. Shorter conversations also produce sharper answers and use fewer tokens.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.get_current_user().strip('"')
my_db = user + '_DB'
my_wh = user + '_WH'
my_role = user + '_RL'
print('Context ready. Signed in as ' + user)
print('Your workspace -> role: ' + my_role + ' | wh: ' + my_wh)

## Set Up Your Workspace
Set your **personal context** — your own role and warehouse — and point reads at the shared course data.

In [ ]:
%%sql -r dataframe_1
-- Set your personal context: your role + your own warehouse
USE ROLE {{my_role}};
CREATE WAREHOUSE IF NOT EXISTS {{my_wh}}
  WAREHOUSE_SIZE = XSMALL AUTO_SUSPEND = 60 AUTO_RESUME = TRUE INITIALLY_SUSPENDED = TRUE;
USE WAREHOUSE {{my_wh}};
-- Read from the shared, read-only course data
USE DATABASE PLATPLUS_DB;
USE SCHEMA ROASTERY_SALES;

## 1 · Two Ways to Get Data

Previously you queried data already in Snowflake. Today's question is *where data comes from in the first place*. It arrives in three shapes:

| Shape | Looks like | Roastery & Co. example |
|---|---|---|
| **Structured** | Rows + columns, fixed schema | `ORDERS`, `STORES` |
| **Semi-structured** | Flexible schema (JSON, Parquet, XML) — stored in a [`VARIANT`](https://docs.snowflake.com/en/user-guide/semistructured-intro) column | Loyalty app events |
| **Unstructured** | No schema at all | Supplier invoice PDFs, product photos |

Before worrying about shape, there's a more fundamental question: **do you need to bring it in at all?**

| | Load it in | Query in place (Marketplace / Sharing) |
|---|---|---|
| **You manage** | A pipeline + storage | Nothing — the provider maintains it |
| **Freshness** | As fresh as your last load | Always provider-current |
| **Best for** | Your own source files | Third-party / partner / reference data |

A publisher's book catalog is a good "query in place" case: the publisher maintains it, you just need to read it. The [**Snowflake Marketplace**](https://docs.snowflake.com/en/user-guide/data-marketplace-intro) and **Private Listings** let you access that data as a live, **zero-copy** share — it appears as a database in your account, but no data is copied and you pay no storage for it.

> :bulb: **The mindset shift:** loading (Lab 7) is one option, not the only one. For data a partner already maintains — a publisher's catalog, a supplier's price list, a vendor's reference feed — mounting a share is far less work than building and babysitting a pipeline.

> 📊 **Why this matters for analysts:** Zero-copy sharing means you get fresh data without waiting for anyone to build a pipeline or refresh a table. You query it like your own data — same SQL, same tools — but you never worry about staleness or storage cost. When a publisher updates their catalog, your next query sees the change automatically. That's one less "can you refresh this?" ticket for you to file.

## 2 · Public Marketplace vs. Private Listing

Snowflake offers two flavors of zero-copy data sharing:

| | Public Marketplace | Private Listing |
|---|---|---|
| **Who can see it** | Anyone on Snowflake | A curated list of invited accounts |
| **How to find it** | Marketplace (top-level) | Data sharing → External sharing → Shared with you |
| **Typical use** | Weather, demographics, financial data | Partner catalogs, internal divisions, supplier feeds |
| **Access** | Click **Get** in the UI | Provider shares directly; you grant access via SQL |

Today's listing — **Alpine Peaks Publishing** — is a **Private Listing**. Alpine Peaks Publishing (a fictional publisher) has shared their forthcoming catalog privately with the Snowflake Education account. You won't find it on the public Marketplace; it lives in your account's **Shared With You** tab.

:point_right: **Navigate to the listing in a new browser tab** (keep this notebook open):
1. In the Snowsight left navigation, **right-click → Open in new tab** on **Data sharing → External sharing** (so you keep this notebook open).
2. Click the **Shared With You** tab.
3. Find and click the **Alpine Peaks Publishing** listing and review its description.

> 💡 **Already provisioned — just browse.** You'll likely see a **"Get Access"** button — that's expected. In a real scenario, clicking it sends a request to your account administrator to grant you access to this shared database. In this lab, Section 4 below runs a stored procedure that acts as the administrator on your behalf and grants your role the necessary privileges. Once that's done, the button changes to **Open**. For now, just review the listing description and note what's included.

> 📚 [Snowflake Marketplace overview](https://docs.snowflake.com/en/user-guide/data-marketplace) · [Private data sharing](https://docs.snowflake.com/en/user-guide/data-sharing-intro)

---

![Alpine Peaks Publishing listing](images/ob_pde_alpine_peaks_listing_1_v2.png)

## 3 · Discover Data on the Public Marketplace

You've seen the *concept* of public vs. private sharing. Now see how you'd actually **find** data on the public **Snowflake Marketplace** — the open catalog any Snowflake account can browse.

**Search it yourself in Snowsight** (open a new browser tab, keep this notebook open):
1. In the left navigation, **right-click → Open in new tab** on **Snowflake Marketplace** (you'll see three options: Snowflake Marketplace, Consumer Hub, and Provider Studio — choose **Snowflake Marketplace**).
2. Use the **search bar** at the top to describe what you need — e.g. `coffee`, `foot traffic`, `consumer spending`.
3. Narrow results with the **Provider**, **Category**, and **Business Need** filters on the left.
4. Click any listing to read its description and sample data. For a free listing, **Get** adds it to your account as a zero-copy share.

> :bulb: Public listings shine for broad **reference data** — weather, demographics, points of interest. Anyone can find and get them. Docs: [Exploring listings on the Marketplace](https://docs.snowflake.com/en/user-guide/data-marketplace-intro).

### Let CoCo drive the search. 🤖
You don't have to click through the UI — CoCo ships two Marketplace skills. Before using them, ask CoCo to explain how they differ (this is a plain question, so **don't** prefix it with a slash — a slash would launch a skill instead of answering):

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">I see two Marketplace skills: /marketplace-search and /get-marketplace-listing-details. How do they differ, and when should I use each one?</pre>

> 🔎 In short: **`/marketplace-search`** casts a wide net and returns a *list* of matching listings; **`/get-marketplace-listing-details`** zooms in on *one* listing you name and explains what's in it and whether it fits your need.

### Search the Marketplace with CoCo. 🤖
Now run a real search for something Roastery-adjacent:

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">/marketplace-search Are there any coffee-related listings available?</pre>

CoCo returns a short list of coffee- and café-adjacent listings, each with a title, a one-line description, and a link. Skim them and notice how varied third-party data can be.

### Zoom in on one listing. 🤖
Pick a listing from those results and get the full picture with the second skill:

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">/get-marketplace-listing-details All Coffee Shop Locations in the Top 5 US Cities</pre>

CoCo returns the listing's provider, what data it contains, and how it might (or might not) fit Roastery & Co.'s needs.

> :point_right: **So why aren't we using a public listing today?** Public listings are perfect for broad reference data — but Roastery needs one specific thing: **Alpine Peaks Publishing's forthcoming catalog**, shared *directly and privately* with this account (the listing you reviewed earlier). You won't surface it by searching the public Marketplace — it's private to us. It's the closest fit for the job, so let's grant access and query it.

## 4 · Get Access to the Shared Database

The `ALPINE_PEAKS_PUBLISHING` database already exists in this account — the instructor mounted it once. What each learner needs is **read access** to it: a non-owner role can't query a shared database until it's granted privileges on the objects inside.

A stored procedure owned by a privileged role handles this in one call, so you don't need `ACCOUNTADMIN`:

:point_right: Run the cell below to grant your `_RL` role access to the shared database.

In [ ]:
%%sql -r dataframe_2
-- Grant your role access to the shared database (one-time, per user)
CALL PLATPLUS_DB.RESOURCES.ACCESS_ALPINE_PEAKS_PUBLISHING_DATA('{{my_role}}');

> 🔎 **What just happened.** The procedure granted your role **`IMPORTED PRIVILEGES`** on the shared database — the standard consumer-side way to open a database created from a share. When you inspect your grants you'll see them surface as targeted rows — `USAGE` on the database, `USAGE` on the `BOOKS` schema, `SELECT` on the `FORTHCOMING_RELEASES` table — because that's exactly what the **provider** chose to put in the share. You can read that one table, nothing more.

> :bulb: **How you open a share — `IMPORTED PRIVILEGES`.** A consumer can't hand out object-level grants on a shared database — Snowflake blocks `GRANT SELECT` on shared objects. The one lever is `GRANT IMPORTED PRIVILEGES ON DATABASE … TO ROLE`, which passes through exactly the objects the **provider** placed in the share (here, one table). Want finer control? The **provider** can define **database roles** to segment the share (Alpine Peaks hasn't), or you can build a **view** over the shared table in your own database and grant on that. The provider controls *what's* in the share; you control *who* in your account can reach it.

> 📊 **Lab vs. real world:** The Education Services team pre-built this procedure so you can grant yourself access during the lab — you just needed to call it. In production, a Platform Admin (or Data Engineer) would handle provisioning shared databases and granting your role access. What you'd see as the analyst is simply a **read-only database** appearing in your account — same SQL, same tools, no different from querying any other table. Your role in the real-world workflow: browse the Marketplace, discover data that supports your analysis, and flag useful listings to your admin. They handle provisioning; you handle the insights.

## 5 · Inspect the Shared Database

A shared database behaves like any other: it has schemas and tables you can explore with `SHOW` and `SELECT`. Run the three cells below to get oriented before you query.

In [ ]:
%%sql -r dataframe_3
-- What schemas are in the shared database?
SHOW SCHEMAS IN DATABASE ALPINE_PEAKS_PUBLISHING;

In [ ]:
%%sql -r dataframe_4
-- What tables are available?
SHOW TABLES IN DATABASE ALPINE_PEAKS_PUBLISHING;

In [ ]:
%%sql -r dataframe_5
-- Preview the forthcoming releases catalog
SELECT *
FROM ALPINE_PEAKS_PUBLISHING.BOOKS.FORTHCOMING_RELEASES
LIMIT 10;

> 🔎 **Note the columns.** The table has `TITLE`, `AUTHOR`, `CATEGORY`, and `RELEASE_DATE` — a clean structured catalog of upcoming book releases. No joins needed to start exploring.

## 6 · Explore the Catalog and Build a Reusable View

The café manager wants to know what **Cooking** titles are coming this year so they can plan when to stock the shelves. Work through the queries below: start broad, then narrow — then save the result as a reusable view.

In [ ]:
%%sql -r dataframe_6
-- How many titles are in the catalog?
SELECT COUNT(*)
FROM ALPINE_PEAKS_PUBLISHING.BOOKS.FORTHCOMING_RELEASES;

In [ ]:
%%sql -r dataframe_7
-- What genres does this publisher cover?
SELECT DISTINCT CATEGORY
FROM ALPINE_PEAKS_PUBLISHING.BOOKS.FORTHCOMING_RELEASES
ORDER BY 1;

In [ ]:
%%sql -r dataframe_8
-- Which genres have the most forthcoming titles?
SELECT CATEGORY, COUNT(*) AS title_count
FROM ALPINE_PEAKS_PUBLISHING.BOOKS.FORTHCOMING_RELEASES
GROUP BY CATEGORY
ORDER BY title_count DESC;

### Package the café shortlist as a view. 🤖
Instead of re-typing the Cooking query every time the café manager asks, save it once as a **view** — a reusable object that stores only its definition and reads the publisher's live data on every query (still zero-copy, still no storage).

The cell below already has the `CREATE OR REPLACE VIEW … AS` line. Ask CoCo for the **SELECT body** that goes under it:

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">@ALPINE_PEAKS_PUBLISHING Don't run it — write just the SELECT statement (no CREATE, no trailing semicolon) that returns all Cooking titles — TITLE, AUTHOR, CATEGORY, RELEASE_DATE — from ALPINE_PEAKS_PUBLISHING.BOOKS.FORTHCOMING_RELEASES ordered by release_date ascending. I'll paste it as the body of a view.</pre>

> :point_right: Paste CoCo's SELECT **after** the CREATE line, then run the cell. Reference solution is just below.

In [ ]:
%%sql -r dataframe_9
CREATE OR REPLACE VIEW {{my_db}}.PUBLIC.CAFE_COOKING_SHORTLIST AS
-- Paste CoCo's SELECT statement on the blank line below, then run this cell


<details>
<summary>📖 Reference solution</summary>

    CREATE OR REPLACE VIEW {{my_db}}.PUBLIC.CAFE_COOKING_SHORTLIST AS
    SELECT TITLE, AUTHOR, CATEGORY, RELEASE_DATE
    FROM ALPINE_PEAKS_PUBLISHING.BOOKS.FORTHCOMING_RELEASES
    WHERE CATEGORY = 'Cooking'
    ORDER BY RELEASE_DATE;

</details>

In [ ]:
%%sql -r dataframe_10
-- Query your new view — a reusable, zero-copy window onto the publisher's Cooking titles
SELECT * FROM {{my_db}}.PUBLIC.CAFE_COOKING_SHORTLIST;


> 🔎 **Review the output.** `CAFE_COOKING_SHORTLIST` is now a saved object in your own database — but it stores **no rows**. Every time it's queried it reads Alpine Peaks' live catalog. That's a reusable, zero-copy interface: downstream dashboards and models point at your view, not the raw share.

> :bulb: **On the `ORDER BY`.** We keep `ORDER BY` in the view here for readability, so the shortlist reads in release order. In production, views usually **omit** `ORDER BY` — downstream queries re-sort anyway, so sorting inside the view is wasted work. Handy to know as you build more of these.

> :bulb: **The business connection.** Roastery & Co. can now plan their café stock around an external data source they didn't build or load — and hand analysts a clean, stable view instead of a raw shared table. In real scenarios this pattern covers supplier price lists, partner inventory feeds, or any dataset a collaborator already maintains and shares privately.

## Check Your Work
Run the next cell to confirm your role can reach the shared database. Prints a ✅ / ⛔ for each check — no need to read the code.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.get_current_user().strip('"')
my_db = user + '_DB'
checks = [
    ("You can reach the shared Roastery data",
     "SELECT COUNT(*) FROM PLATPLUS_DB.ROASTERY_SALES.STORES", 1),
    ("Your role can query the Alpine Peaks Publishing catalog",
     "SELECT COUNT(*) FROM ALPINE_PEAKS_PUBLISHING.BOOKS.FORTHCOMING_RELEASES", 1),
    ("Your CAFE_COOKING_SHORTLIST view returns Cooking titles",
     f"SELECT COUNT(*) FROM {my_db}.PUBLIC.CAFE_COOKING_SHORTLIST", 1),
]
passed = 0
for label, sql, need in checks:
    try:
        n = session.sql(sql).collect()[0][0]
        ok = n >= need
    except Exception:
        n, ok = None, False
    passed += 1 if ok else 0
    icon = '✅' if ok else '⛔'
    detail = f'(found {n})' if n is not None else '(not accessible — re-run the steps above)'
    print(f'{icon}  {label}  {detail}')
print('-' * 52)
if passed == len(checks):
    print(f'🎉 ALL {len(checks)} CHECKS PASSED — lab complete! Nice work.')
else:
    print(f'⚠️  {passed}/{len(checks)} passed — re-run the cells above for any ⛔ item, then run this cell again.')

### Check your understanding. 🧠
A few quick questions on this lab's key themes. Run each cell below, choose an answer, and the feedback appears inline — no grade, just a self-check.

In [ ]:
from resources.knowledgecheck import data_analyst_quiz as quiz
quiz.display_question("q1")

In [ ]:
from resources.knowledgecheck import data_analyst_quiz as quiz
quiz.display_question("q2")

In [ ]:
from resources.knowledgecheck import data_analyst_quiz as quiz
quiz.display_question("q3")

### Key takeaways. ❄️
- **Zero copy** means no data moves — a shared database is a live window onto the provider's data, no storage cost on your side
- The **public Marketplace** is self-serve — search it in Snowsight, or drive it with CoCo's `/marketplace-search` (find listings) and `/get-marketplace-listing-details` (inspect one)
- **Private Listings** are for curated, account-specific data sharing — not visible on the public Marketplace
- Getting access = **`IMPORTED PRIVILEGES`** on the shared database — the one consumer-side lever; it surfaces exactly the objects the provider shared (here, one table). Finer control comes from the **provider** (database roles) or from a **view** you build over the share
- A **view over a shared table** (`CAFE_COOKING_SHORTLIST`) is a reusable, zero-copy interface — it stores no rows, and downstream users query the view instead of the raw share
- Decide per dataset: **load it in** (your own source files) vs. **query in place** (partner / reference data)

> 🚀 **Coming up in the capstone:** you'll bring datasets together and apply AI functions to turn raw data into a decision — the kind of enrichment that makes shared and Marketplace data truly valuable.

### Notebook completion. 🏁
You're done — your notebook is saved automatically.

> 💡 **Moving on to another lab?** Shut down this notebook's **kernel** — click **Connected** at the top → **Shut down kernel**. All your lab notebooks share one compute service, so clearing a finished kernel frees up memory for the next lab. Your service stays running, ready in seconds.

> 💡 **Finished for the day?** **Suspend your service** — click **Connected** → hover your service name → **Suspend**. This stops compute; it resumes automatically next time you run a cell.